# Day 210 — Week 3, Day 3: Before/After Evaluation with Real Numbers (NRA)
### Month 13: Agentic AI & Advanced GenAI Portfolio | TeleServe India Ticket Priority Classifier

**Builds on:** Day208 (LoRA/QLoRA setup) + Day209 (QLoRA training run, adapter saved at `./teleserve_lora_adapter_209`)

**Today's focus:** Day209 gave you ONE before/after number pair (33.33% → 100%) on the **validation set** — and one honest red flag (all 3 hand-written test tickets predicted the same class). Today you build a **proper evaluation report**: multiple metrics, multiple test sets, and an NRA write-up that a client or interviewer would actually trust.

**Total: 100 points + 10 bonus**


---
## Section 0 — Concept Notes (What & Why)

### What "before/after evaluation" actually means
Day209 answered: *"did fine-tuning improve accuracy?"* Today answers the harder, more useful question: *"improve accuracy **on what**, and **can I trust that number**?"*

A single accuracy number on a single validation split is not an evaluation — it's a data point. A real evaluation needs:

| Concept | Why it matters |
|---|---|
| **Multiple metrics** (precision, recall, F1 per class) | Accuracy hides class-level failure. A model that is 100% accurate on P3 but 0% on P1 (the *critical* class) is dangerous in production, and overall accuracy won't show you that. |
| **Multiple test sets** | Val accuracy tells you if the model learned the *training distribution*. A NEW held-out templated set tells you if it generalizes *within* that distribution. A hand-written, non-templated set tells you if it generalizes to *real customer language* — the only one that matters in production. |
| **Confusion matrix** | Shows *which* classes get confused with which — critical for a priority classifier, since confusing P1↔P3 is a much worse failure than confusing P2↔P3. |
| **Before vs After, same test sets** | The baseline (untrained head) and fine-tuned model must be scored on the *exact same* test sets, or the comparison is meaningless. |

### Why this day exists (the honest reason)
Day209's Bonus task already surfaced a red flag: the fine-tuned model predicted **P3 for all 3** hand-written tickets despite 100% validation accuracy. That is not a bug — it's the single most important lesson in applied ML: **validation accuracy measures memorization of the training distribution, not real-world generalization, unless your validation set itself is diverse.** Today quantifies exactly how large that gap is, with real numbers, across a properly sized (30-example) hard set instead of 3 anecdotal examples.

### The real-world/client framing
A client who reads "100% accuracy" in a report and then watches the model fail on the first real ticket will never trust your work again. A client who reads *"100% on templated data, 40% on real customer phrasing — here's why, and here's the fix"* trusts you **more**, because you caught it before they did. This is the difference between a report and a **liability**.

### NRA reminder (applies to every metric you report today)
- **Number**: read directly from printed `classification_report` / `confusion_matrix` output — never estimated, never rounded from memory.
- **Reason**: state the *mechanism* (why the number is what it is) — not the outcome restated, not what to do about it.
- **Action**: name a specific next step, threshold, or owner — no hedging like "should investigate further."


---
## Section 1 — Raw Data (DO NOT MODIFY)

Three evaluation assets, all seeded and reproducible:
1. **Val set** — reuse Day209's held-out val split (90 tickets, templated, same distribution as training) — this is the number you already have from Day209.
2. **NEW templated test set** (`test_templated_210`) — 90 tickets (30/class), generated fresh below with a **different template wording pool** than Day208/209 training used. Tests generalization *within* the templated style.
3. **Hard test set** (`hard_test_210`) — 30 hand-written tickets (10/class), natural customer phrasing, no template structure at all. This is the set that matters most for production trust.

Run the cell below once. Treat its output as fixed raw data for the rest of the notebook — do not hand-edit the lists afterward.


In [1]:
import random
random.seed(210)

# TeleServe India - Day210 NEW held-out templated test set
# Priority classes: P1 (Critical), P2 (Moderate), P3 (Low)
# NOTE: wording pool is deliberately DIFFERENT from Day208/209 training templates,
# so this set tests generalization to new phrasing, not just a train/val re-split.

customers = ["Rohan Sharma","Priya Nair","Arjun Mehta","Sneha Iyer","Vikram Shah","Ananya Rao",
             "Karan Malhotra","Divya Menon","Rahul Verma","Isha Kapoor","Aditya Joshi","Neha Reddy",
             "Suresh Pillai","Kavya Nair","Manoj Tiwari","Pooja Desai"]
cities = ["Rohtak","Gurugram","Pune","Chennai","Hyderabad","Lucknow","Jaipur","Indore","Bhopal","Nagpur"]

p1_templates = [
    "Complete service outage in {city} since {hrs} hours, no dial tone on landline for {name}.",
    "{name} reports total network blackout affecting entire {city} sector, business operations halted.",
    "Fiber connection completely down for {name} in {city}, zero connectivity for {hrs}+ hours.",
    "Security breach suspected on {name}'s account in {city} branch, unauthorized SIM swap alert triggered.",
    "Emergency: {name}'s primary business line dead in {city}, unable to receive customer calls at all.",
    "Full mobile network failure reported by {name} across {city} region, no signal bars showing.",
]
p2_templates = [
    "{name} in {city} experiencing intermittent call drops roughly every {hrs} minutes on landline.",
    "Internet speed for {name} in {city} has dropped to under 2mbps from the subscribed plan.",
    "Billing discrepancy of significant amount flagged by {name}, {city} account overcharged last cycle.",
    "{name} reports patchy 4G signal in {city}, works outdoors but fails indoors consistently.",
    "Recurring one-way audio issue on calls for {name} in {city}, caller can't hear response.",
    "{name}'s broadband disconnects randomly {hrs} times a day in {city}, needs router reset each time.",
]
p3_templates = [
    "{name} from {city} requesting information about upgrading to the family data plan.",
    "General query from {name} in {city} about international roaming rates for upcoming trip.",
    "{name} wants to know how to change autopay date on {city} account, no urgency.",
    "Minor complaint from {name} in {city} about SMS notification wording being unclear.",
    "{name} asking for a duplicate invoice copy for {city} account, tax filing purpose.",
    "{name} in {city} curious about new 5G rollout timeline, no current service issue.",
]

def make_rows(templates, label, n):
    rows = []
    for i in range(n):
        t = random.choice(templates)
        name = random.choice(customers)
        city = random.choice(cities)
        hrs = random.randint(2, 8)
        text = t.format(name=name, city=city, hrs=hrs)
        rows.append((text, label))
    return rows

test_templated_210 = make_rows(p1_templates, "P1", 30) + make_rows(p2_templates, "P2", 30) + make_rows(p3_templates, "P3", 30)
random.shuffle(test_templated_210)

print(f"test_templated_210: {len(test_templated_210)} tickets")
from collections import Counter
print(Counter([l for _, l in test_templated_210]))


test_templated_210: 90 tickets
Counter({'P3': 30, 'P2': 30, 'P1': 30})


In [2]:
# Hand-written, non-templated hard test set - 10 per class, natural customer phrasing
# NOT generated from the p1/p2/p3 template pool above - mimics real customer typing style

hard_test_210 = [
    # P1 - Critical (10)
    ("my whole office in gurugram has zero network since this morning nobody can call us this is costing us clients", "P1"),
    ("URGENT the fiber line to our branch just went completely dead we have a client demo in an hour", "P1"),
    ("someone hacked into my account and ported my number without my permission please help immediately", "P1"),
    ("landline has been silent for two days now, not even a dial tone, this is our only business number", "P1"),
    ("entire building lost signal after the storm last night, 40 employees can't make a single call", "P1"),
    ("getting an unauthorized sim activation alert on my phone right now, i think my line is being stolen", "P1"),
    ("our server room connectivity dropped to nothing, production systems are offline because of this", "P1"),
    ("no network bars anywhere in the office since 6am, tried restarting everything, nothing works", "P1"),
    ("the outage from yesterday never got fixed, we still have absolutely no service at the warehouse", "P1"),
    ("critical - hospital wing phones are all dead, doctors can't be reached, need this fixed now", "P1"),
    # P2 - Moderate (10)
    ("calls keep cutting off after a couple minutes every single time, super annoying but manageable", "P2"),
    ("been charged almost double what i usually pay this month, can someone check my bill", "P2"),
    ("wifi is really slow lately, videos buffer constantly even though nothing changed on my end", "P2"),
    ("signal is fine outside but the moment i step into my apartment it drops to one bar", "P2"),
    ("people tell me they can hear me but i can't hear them half the time on calls", "P2"),
    ("router just disconnects on its own a few times a day, i have to unplug and plug back in", "P2"),
    ("noticed an extra charge on my statement i don't recognize, want it explained please", "P2"),
    ("data speeds have been crawling this week, definitely not the 4G i'm paying for", "P2"),
    ("call quality is choppy and robotic sounding during peak evening hours only", "P2"),
    ("my broadband uptime feels worse than usual, drops maybe five or six times daily now", "P2"),
    # P3 - Low (10)
    ("just wondering what the roaming charges look like if i travel to dubai next month", "P3"),
    ("is there a way to switch my bill payment date to the 15th instead of the 1st", "P3"),
    ("curious if you guys have any family bundle plans i could switch to", "P3"),
    ("can you send me last month's invoice again, i misplaced the copy for my records", "P3"),
    ("the text alerts i get are a bit confusing worded, might want to simplify them sometime", "P3"),
    ("when is 5G actually coming to our area, just curious no complaints otherwise", "P3"),
    ("thinking about upgrading my data plan, what options do i have right now", "P3"),
    ("no issues really, just want to understand how the loyalty points program works", "P3"),
    ("do you offer any student discounts on postpaid plans", "P3"),
    ("everything's working fine, just checking if there's a newer phone upgrade offer available", "P3"),
]

print(f"hard_test_210: {len(hard_test_210)} tickets")
print(Counter([l for _, l in hard_test_210]))


hard_test_210: 30 tickets
Counter({'P1': 10, 'P2': 10, 'P3': 10})


---
## Section 2 — Practice Guide (Tasks)

**Setup reminder:** You need Day209's saved artifacts loaded first — `teleserve_lora_adapter_209/` (the trained adapter) and the label encoding (`P1`→0, `P2`→1, `P3`→2, or whatever mapping you used Day209 — check your Day209 notebook if unsure). Runtime → T4 GPU, same as Day209.


In [3]:
# --- Setup cell: install/import (same stack as Day209) ---
!pip install -q transformers peft accelerate bitsandbytes scikit-learn --upgrade
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121 --upgrade

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA available: True
GPU: Tesla T4


### Task 1 — Load baseline (before) and fine-tuned (after) models (15 pts)

Load TWO models:
1. **Baseline**: fresh `distilbert-base-uncased` with a randomly-initialized 3-class head (same as Day209's "BEFORE" model — untrained on this task).
2. **Fine-tuned**: the SAME base model with Day209's saved LoRA adapter loaded on top (`PeftModel.from_pretrained`).

Both must use the identical tokenizer and label mapping so predictions are comparable.

**TODO:**


In [4]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.11.0+cu128
12.8
True


In [9]:
# ── TASK 1: Load baseline and fine‑tuned models ──────────────────────
# Goal: Instantiate the untrained baseline (random head) and the LoRA‑fine‑tuned
#       model loaded from the saved adapter (with 4‑bit quantization matching Day 209).
# Method: Define label mapping, load tokenizer, load fresh DistilBERT for baseline,
#         then load the same base with BitsAndBytesConfig via device_map="auto"
#         (which already places quantized weights on GPU correctly — no extra .cuda() needed).

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from google.colab import drive

# Mount Google Drive if adapter is stored there
drive.mount('/content/drive')

ADAPTER_PATH = "/content/drive/MyDrive/teleserve_lora_adapter_209"   # adjust if needed
BASE_MODEL = "distilbert-base-uncased"

label2id = {"P1": 0, "P2": 1, "P3": 2}
id2label = {v: k for k, v in label2id.items()}

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# --- 1. Baseline (before) — plain fp32, .cuda() is correct and needed here ---
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
baseline_model.cuda()
baseline_model.eval()
torch.cuda.empty_cache()

# --- 2. Fine‑tuned (after) ---
# Quantization config exactly as in Day 209 training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["pre_classifier", "classifier"],  # keep the trained head in full precision
)

finetuned_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    quantization_config=bnb_config,
    device_map="auto",
)

finetuned_base = prepare_model_for_kbit_training(finetuned_base)
finetuned_model = PeftModel.from_pretrained(finetuned_base, ADAPTER_PATH)
finetuned_model.eval()
torch.cuda.empty_cache()

print("✅ Models loaded and quant states initialized.")
print(f"Fine‑tuned device: {next(finetuned_model.parameters()).device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider t

✅ Models loaded and quant states initialized.
Fine‑tuned device: cuda:0


### Task 2 — Build a reusable prediction + metrics function (20 pts)

Write ONE function `evaluate_model(model, dataset, set_name)` that:
1. Tokenizes and runs inference on every example in `dataset` (a list of `(text, label)` tuples)
2. Collects true labels and predicted labels
3. Prints `accuracy_score`, full `classification_report` (digits=4), and `confusion_matrix` with labeled axes
4. Returns a dict: `{"set": set_name, "accuracy": ..., "report": ..., "confusion": ...}`

You will call this function 6 times total (2 models × 3 test sets) — writing it once and reusing it is the point, not copy-pasting inference code 6 times.

**TODO:**


In [10]:
# ── TASK 2: Reusable evaluate_model function ──────────────────────────
# Goal: Write a single function that computes accuracy, classification report,
#       and confusion matrix for any model and dataset.
# Method: Tokenize each text in batches, run model inference, collect true/pred labels,
#         compute metrics via sklearn functions, and print/return results.

def evaluate_model(model, dataset, set_name, batch_size=16):
    """
    Evaluate a model on a dataset of (text, label) tuples.
    Returns a dict with set_name, accuracy, report (string), confusion (list of lists),
    and also prints the report and confusion matrix.
    """
    texts, labels = zip(*dataset)
    y_true = [label2id[l] for l in labels]
    y_pred = []

    # Process in batches to avoid OOM
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(list(batch_texts), truncation=True, padding=True, max_length=64, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
        preds = logits.argmax(dim=-1).cpu().numpy()
        y_pred.extend(preds)

    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=["P1","P2","P3"], digits=4)
    conf = confusion_matrix(y_true, y_pred)

    print(f"\n=== {set_name} ===")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix (rows=true, cols=pred):")
    print("       P1   P2   P3")
    for i, row in enumerate(conf):
        print(f"P{i+1}  {row[0]:>4} {row[1]:>4} {row[2]:>4}")

    return {
        "set": set_name,
        "accuracy": acc,
        "report": report,
        "confusion": conf.tolist()
    }

### Task 3 — Run all 6 evaluations and build the comparison table (20 pts)

Run `evaluate_model` for:
- Baseline on val set (reuse Day209's val split — reload it or regenerate with the same seed/split logic)
- Baseline on `test_templated_210`
- Baseline on `hard_test_210`
- Fine-tuned on val set
- Fine-tuned on `test_templated_210`
- Fine-tuned on `hard_test_210`

Then build a single pandas DataFrame comparing all 6 accuracy numbers side by side (rows = test set, columns = Before / After / Delta).

**TODO:**


In [11]:
# ── TASK 3: Regenerate Day209 validation split ──────────────────────
# Goal: Obtain the 90‑ticket validation set used in Day209 for fair comparison.
# Method: Load the raw CSV, map priorities to labels, and split with
#         train_test_split(test_size=0.2, stratify=labels, random_state=209)
#         exactly as in Day209.

import pandas as pd
from sklearn.model_selection import train_test_split

# Load raw data – this file is in the same directory
raw_df = pd.read_csv("teleserve_tickets_209.csv")

# Map priority to label
raw_df["label"] = raw_df["priority"].map(label2id)

# Stratified split with seed 209 (same as Day209)
train_df, val_df = train_test_split(
    raw_df,
    test_size=0.2,
    stratify=raw_df["label"],
    random_state=209
)

# Create list of (text, priority) tuples for evaluation
val_set_209 = list(zip(val_df["text"], val_df["priority"]))

print(f"Validation set size: {len(val_set_209)}")
print("Class balance:", val_df["priority"].value_counts().to_dict())

Validation set size: 90
Class balance: {'P2': 30, 'P1': 30, 'P3': 30}


In [12]:
# ── TASK 3: Run evaluations and build comparison table ────────────────
# Goal: Compute accuracy for baseline and fine‑tuned on all three test sets.
# Method: Call evaluate_model for each combination, store results in a dict,
#         then create a pandas DataFrame with Before/After/Delta.

# Ensure val_set_209 is defined; if not, skip or raise error.
if val_set_209 is None:
    print("ERROR: val_set_209 is None. Please populate it with your validation data.")
    # For demonstration, we'll use the templated test set as a fallback (not correct).
    # In real usage, you should fix this.
    val_set_209 = test_templated_210[:90]  # placeholder - NOT correct for actual validation

results = {}

# Baseline evaluations
results["baseline_val"] = evaluate_model(baseline_model, val_set_209, "Val (baseline)")
results["baseline_templated"] = evaluate_model(baseline_model, test_templated_210, "New Templated (baseline)")
results["baseline_hard"] = evaluate_model(baseline_model, hard_test_210, "Hard/Non-templated (baseline)")

# Fine-tuned evaluations
results["finetuned_val"] = evaluate_model(finetuned_model, val_set_209, "Val (fine-tuned)")
results["finetuned_templated"] = evaluate_model(finetuned_model, test_templated_210, "New Templated (fine-tuned)")
results["finetuned_hard"] = evaluate_model(finetuned_model, hard_test_210, "Hard/Non-templated (fine-tuned)")

# Build comparison DataFrame
comparison_data = {
    "Test Set": ["Val", "New Templated", "Hard/Non-templated"],
    "Before (%)": [
        results["baseline_val"]["accuracy"] * 100,
        results["baseline_templated"]["accuracy"] * 100,
        results["baseline_hard"]["accuracy"] * 100,
    ],
    "After (%)": [
        results["finetuned_val"]["accuracy"] * 100,
        results["finetuned_templated"]["accuracy"] * 100,
        results["finetuned_hard"]["accuracy"] * 100,
    ],
}
comparison_df = pd.DataFrame(comparison_data)
comparison_df["Delta (pts)"] = comparison_df["After (%)"] - comparison_df["Before (%)"]

print("\n=== Comparison Table ===")
print(comparison_df)


=== Val (baseline) ===
Accuracy: 0.1778 (17.78%)
Classification Report:
              precision    recall  f1-score   support

          P1     0.0000    0.0000    0.0000        30
          P2     0.2083    0.1667    0.1852        30
          P3     0.1746    0.3667    0.2366        30

    accuracy                         0.1778        90
   macro avg     0.1276    0.1778    0.1406        90
weighted avg     0.1276    0.1778    0.1406        90

Confusion Matrix (rows=true, cols=pred):
       P1   P2   P3
P1     0    0   30
P2     3    5   22
P3     0   19   11

=== New Templated (baseline) ===
Accuracy: 0.3778 (37.78%)
Classification Report:
              precision    recall  f1-score   support

          P1     0.2083    0.1667    0.1852        30
          P2     0.0000    0.0000    0.0000        30
          P3     0.4462    0.9667    0.6105        30

    accuracy                         0.3778        90
   macro avg     0.2182    0.3778    0.2652        90
weighted avg     0.

### Task 4 — Error analysis on the hard test set (15 pts)

The hard test set is where production trust is actually won or lost. For the **fine-tuned** model's predictions on `hard_test_210`:
1. Print every misclassified example: the text, true label, predicted label.
2. Count how many errors are P1→P3 or P1→P2 (a critical ticket mis-routed as non-critical) vs other error types.
3. State in one sentence why a P1→P3 error is worse for a business than a P2→P3 error, even though both count equally toward "accuracy."

**TODO:**


In [13]:
# ── TASK 4: Error analysis on hard_test_210 for fine‑tuned model ──
# Goal: Print misclassified examples, tally error types, and explain severity.
# Method: Loop through hard_test_210, get predictions, compare to true labels.

texts, labels = zip(*hard_test_210)
y_true = [label2id[l] for l in labels]
y_pred = []

batch_size = 16
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    inputs = tokenizer(list(batch_texts), truncation=True, padding=True, max_length=64, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        logits = finetuned_model(**inputs).logits
    preds = logits.argmax(dim=-1).cpu().numpy()
    y_pred.extend(preds)

error_count = 0
error_types = Counter()

print("\n=== Misclassified examples (fine-tuned on hard set) ===")
for idx, (text, true_label) in enumerate(hard_test_210):
    true_id = label2id[true_label]
    pred_id = y_pred[idx]
    if true_id != pred_id:
        error_count += 1
        error_types[(true_label, id2label[pred_id])] += 1
        print(f"{idx+1}. TRUE: {true_label}  PRED: {id2label[pred_id]}")
        print(f"   Text: {text}\n")

print(f"\nTotal errors: {error_count} out of {len(hard_test_210)}")
print("Error type counts (true → pred):")
for (true, pred), count in error_types.most_common():
    print(f"  {true} → {pred}: {count}")

print("\n=== Severity reasoning ===")
print("A P1→P3 error is more severe than a P2→P3 error because a critical ticket (P1) misclassified as low priority (P3) will be ignored, potentially causing severe business impact (e.g., lost clients, safety risks), whereas a moderate ticket (P2) misclassified as low still receives some attention, albeit delayed.")


=== Misclassified examples (fine-tuned on hard set) ===
2. TRUE: P1  PRED: P3
   Text: URGENT the fiber line to our branch just went completely dead we have a client demo in an hour

3. TRUE: P1  PRED: P3
   Text: someone hacked into my account and ported my number without my permission please help immediately

4. TRUE: P1  PRED: P3
   Text: landline has been silent for two days now, not even a dial tone, this is our only business number

6. TRUE: P1  PRED: P3
   Text: getting an unauthorized sim activation alert on my phone right now, i think my line is being stolen

9. TRUE: P1  PRED: P2
   Text: the outage from yesterday never got fixed, we still have absolutely no service at the warehouse

11. TRUE: P2  PRED: P3
   Text: calls keep cutting off after a couple minutes every single time, super annoying but manageable

12. TRUE: P2  PRED: P3
   Text: been charged almost double what i usually pay this month, can someone check my bill

13. TRUE: P2  PRED: P3
   Text: wifi is really slow

### Task 5 — NRA Evaluation Report (30 pts)

Write a **client-facing evaluation summary** (markdown cell) using strict Number + Reason + Action format. This is the deliverable a client actually reads — the tables above are your working evidence, this is the conclusion.

Cover, in three separate NRA blocks:
1. **The generalization gap** — the accuracy delta between templated sets and the hard set, for the fine-tuned model.
2. **The class-level risk** — whichever class had the worst recall on the hard set (read it from Task 3's classification_report, don't guess).
3. **Whether this model is production-ready as-is** — a direct recommendation with a threshold-based justification.

Reminder of NRA discipline (this has been the recurring gap all quarter):
- **Number**: copied verbatim from printed output above, not estimated.
- **Reason**: states *why* the number is what it is (mechanism) — one sentence, then STOP. Do not restate the number, do not slide into what to do about it.
- **Action**: specific, owned, time-boxed. No "should investigate further" — say what, by whom, by when, and what threshold triggers it.

**TODO (markdown):**


# ── TASK 5: NRA Evaluation Report ────────────────────────────────────

## 1. Generalization gap
**Number:** The fine‑tuned model achieved **100.00%** accuracy on the validation set, **60.00%** on the new templated set, and **60.00%** on the hand‑written hard set.  
**Reason:** The model overfits to the template structures seen during training; it memorizes phrase patterns rather than learning semantic urgency.  
**Action:** Before production, collect at least 200 real customer tickets from the last 6 months, label them with a human panel, and use them as a new test set. Retrain only if the accuracy on that set exceeds 80%; otherwise, augment training data with paraphrased templates and real examples.

## 2. Class‑level risk
**Number:** On the hard set, class **P2** had the lowest recall: **30.00%** (from the classification_report above).  
**Reason:** One plausible mechanism is that P2 training examples may have been more template‑dependent than the other classes, leaving the model unprepared for the varied vocabulary and sentence structures in real moderate‑severity complaints.  
**Action:** Within 2 weeks, generate 100 synthetic P2 tickets from real‑world incident descriptions (e.g., from billing and connectivity forums) and add them to the training set. Then re‑evaluate on the same hard set; if recall for P2 does not improve by at least 10 percentage points, schedule a manual rule‑based fallback for that class.

## 3. Production‑readiness recommendation
**Number:** The model is **NOT** production‑ready as‑is, because its hard‑set accuracy of **60.00%** is substantially lower than its validation accuracy of 100.00%, and it makes 5 critical‑class errors (4 P1→P3 + 1 P1→P2) that would route urgent tickets to low‑priority queues.  
**Reason:** High validation accuracy on templated data does not translate to real‑world performance; the model’s errors on hard tickets would directly harm customers and the business.  
**Action:** Do not deploy this model as an auto‑classifier. Instead, use it as a triage aid: route any ticket with predicted confidence < 0.6 to a human agent (see Bonus), and only auto‑classify tickets with high confidence (>0.9) after verifying on a held‑out real dataset. Re‑evaluate in 4 weeks after incorporating the additional training data mentioned above.

### Bonus (+10 pts) — Confidence-based triage

For the fine-tuned model on `hard_test_210`, extract the softmax probability of the *predicted* class for each example (not just the label). Compare the average confidence on **correct** predictions vs **incorrect** predictions.

Then answer: could a confidence threshold (e.g., "if predicted-class probability < 0.6, route to a human instead of auto-classifying") reduce the real-world risk from today's Task 4 error analysis, without retraining anything? Back your answer with the actual average-confidence numbers you computed.

**TODO:**


In [14]:
# ── BONUS: Confidence analysis on hard set ────────────────────────────
# Goal: Compute softmax probabilities and compare average confidence on correct vs incorrect predictions.
# Method: For each hard test example, get logits, apply softmax, record probability of predicted class.

import torch.nn.functional as F

conf_correct = []
conf_incorrect = []

for text, true_label in hard_test_210:
    inputs = tokenizer([text], truncation=True, padding=True, max_length=64, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        logits = finetuned_model(**inputs).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = np.argmax(probs)
    true_id = label2id[true_label]
    confidence = probs[pred_id]
    if pred_id == true_id:
        conf_correct.append(confidence)
    else:
        conf_incorrect.append(confidence)

avg_conf_correct = np.mean(conf_correct) if conf_correct else 0
avg_conf_incorrect = np.mean(conf_incorrect) if conf_incorrect else 0

print(f"Average confidence on correct predictions: {avg_conf_correct:.4f}")
print(f"Average confidence on incorrect predictions: {avg_conf_incorrect:.4f}")
print(f"Difference: {avg_conf_correct - avg_conf_incorrect:.4f}")

print("\n=== Conclusion ===")
if avg_conf_correct > avg_conf_incorrect + 0.1:
    print("A confidence threshold (e.g., route to human if probability < 0.6) would likely reduce errors because incorrect predictions have notably lower confidence. This can improve safety without retraining.")
else:
    print("Confidence does not clearly separate correct from incorrect predictions; a threshold may not be effective. Consider retraining or using an ensemble.")

Average confidence on correct predictions: 0.8766
Average confidence on incorrect predictions: 0.6401
Difference: 0.2365

=== Conclusion ===
A confidence threshold (e.g., route to human if probability < 0.6) would likely reduce errors because incorrect predictions have notably lower confidence. This can improve safety without retraining.


---
## Interview-Framing Question

*"Walk me through how you evaluated your fine-tuned model before calling it done."*

**Practice your answer** (2-3 sentences, written in your own words — this is what you'd say out loud):

> TODO: e.g. "I didn't stop at validation accuracy, because a validation set drawn from the same templates as training will always look artificially strong. I built two additional test sets — a fresh templated set and a hand-written set mimicking real customer phrasing — and evaluated both the baseline and fine-tuned model on all three with full precision/recall/F1, not just accuracy, so I could see whether errors were concentrated in the safety-critical class."


---
## Section 3 — Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 — Load baseline + fine-tuned models | 15 | Both models load correctly; PeftModel wraps the correct base; tokenizer/label mapping consistent |
| Task 2 — Reusable `evaluate_model` function | 20 | Single function, correctly batches/tokenizes, correct metrics, returns usable dict, no per-call copy-paste |
| Task 3 — 6 evaluations + comparison table | 20 | All 6 runs executed with printed output; `comparison_df` built from `results` dict, not hand-typed numbers |
| Task 4 — Error analysis | 15 | Misclassifications printed; error-type tally correct; P1-vs-P2 severity reasoning is sound (not just "P1 is more important") |
| Task 5 — NRA Evaluation Report | 30 | All 3 Numbers copied from printed output; Reasons state mechanism only (no outcome restatement, no Action bleed); Actions are specific/owned/time-boxed |
| **Bonus — Confidence triage** | +10 | Real confidence numbers computed for correct vs incorrect groups; conclusion is backed by those numbers, not assumed |
| **Total** | **100 (+10)** | |

**Submission standard:** Restart runtime + Run All before submitting. Stale outputs from partial runs are a recurring deduction cause — this applies especially today since Tasks 3–5 all depend on Task 1/2 running cleanly on GPU.
